In [40]:
import sys
sys.path.append("..")

import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from nltk.tokenize import TreebankWordTokenizer

from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
)

from src.config import (
    LOWERCASE,
    MIN_COUNT,
    SEEDS,
    THRESHOLD,
    RESULTS_DIR,
    load_split,
    evaluate,
    save_predictions,
)

In [2]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.1+cpu
CUDA available: False


In [3]:
train_df = load_split("train")
val_df = load_split("validation")

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)

Train: (3668, 4)
Validation: (408, 4)


In [4]:
treebank = TreebankWordTokenizer()

def tokenize(text):
    if LOWERCASE:
        text = text.lower()

    return treebank.tokenize(text)

In [5]:
training_sentences = pd.concat(
    [
        train_df["sentence1"],
        train_df["sentence2"],
    ],
    ignore_index=True,
)

tfidf = TfidfVectorizer(
    tokenizer=tokenize,
    token_pattern=None,
    lowercase=False,
    min_df=MIN_COUNT,
)

tfidf.fit(training_sentences)

print("TF-IDF vocabulary size:", len(tfidf.vocabulary_))

TF-IDF vocabulary size: 9917


In [6]:
train_s1 = tfidf.transform(train_df["sentence1"])
train_s2 = tfidf.transform(train_df["sentence2"])

val_s1 = tfidf.transform(val_df["sentence1"])
val_s2 = tfidf.transform(val_df["sentence2"])

In [8]:
train_diff = abs(train_s1 - train_s2)
val_diff = abs(val_s1 - val_s2)

train_product = train_s1.multiply(train_s2)
val_product = val_s1.multiply(val_s2)

X_train_sparse = hstack(
    [train_diff, train_product]
).tocsr()

X_val_sparse = hstack(
    [val_diff, val_product]
).tocsr()

print("X_train:", X_train_sparse.shape)
print("X_val:", X_val_sparse.shape)

X_train: (3668, 19834)
X_val: (408, 19834)


In [9]:
X_train = torch.tensor(
    X_train_sparse.toarray(),
    dtype=torch.float32,
)

X_val = torch.tensor(
    X_val_sparse.toarray(),
    dtype=torch.float32,
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32,
).unsqueeze(1)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32,
).unsqueeze(1)

print("X_train:", X_train.shape)
print("y_train:", y_train_tensor.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val_tensor.shape)

X_train: torch.Size([3668, 19834])
y_train: torch.Size([3668, 1])
X_val: torch.Size([408, 19834])
y_val: torch.Size([408, 1])


In [10]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

In [11]:
SEEDS = [42, 43, 44]

In [12]:
seed = SEEDS[0]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [13]:
input_dim = X_train.shape[1]

model = FeedForwardNN(input_dim)

print(model)

FeedForwardNN(
  (network): Sequential(
    (0): Linear(in_features=19834, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [14]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [15]:
train_dataset = TensorDataset(
    X_train,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

print("Number of batches:", len(train_loader))

Number of batches: 58


In [16]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        # 1. Remove gradients left from previous batch
        optimizer.zero_grad()

        # 2. Forward pass
        logits = model(X_batch)

        # 3. Calculate prediction error
        loss = criterion(logits, y_batch)

        # 4. Backpropagation
        loss.backward()

        # 5. Update model weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1:2d}/{EPOCHS} "
        f"- Training Loss: {average_loss:.4f}"
    )

Epoch  1/10 - Training Loss: 0.6582
Epoch  2/10 - Training Loss: 0.5305
Epoch  3/10 - Training Loss: 0.2685
Epoch  4/10 - Training Loss: 0.0900
Epoch  5/10 - Training Loss: 0.0298
Epoch  6/10 - Training Loss: 0.0153
Epoch  7/10 - Training Loss: 0.0095
Epoch  8/10 - Training Loss: 0.0068
Epoch  9/10 - Training Loss: 0.0053
Epoch 10/10 - Training Loss: 0.0050


In [18]:
seed = SEEDS[0]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model = FeedForwardNN(X_train.shape[1])

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [19]:
EPOCHS = 10

history = []

for epoch in range(EPOCHS):
    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    model.eval()

    with torch.no_grad():

        val_logits = model(X_val)

        val_probs = torch.sigmoid(val_logits).squeeze(1)

        val_preds = (
            val_probs >= THRESHOLD
        ).int().numpy()

    val_scores = evaluate(
        y_val,
        val_preds
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_accuracy": val_scores["accuracy"],
        "val_f1": val_scores["f1"],
    })

    print(
        f"Epoch {epoch + 1:2d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_scores['accuracy']:.4f} | "
        f"Val F1: {val_scores['f1']:.4f}"
    )

Epoch  1/10 | Loss: 0.6582 | Val Acc: 0.6912 | Val F1: 0.8158
Epoch  2/10 | Loss: 0.5305 | Val Acc: 0.7279 | Val F1: 0.8257
Epoch  3/10 | Loss: 0.2685 | Val Acc: 0.7255 | Val F1: 0.8164
Epoch  4/10 | Loss: 0.0900 | Val Acc: 0.7304 | Val F1: 0.8208
Epoch  5/10 | Loss: 0.0298 | Val Acc: 0.6985 | Val F1: 0.7868
Epoch  6/10 | Loss: 0.0153 | Val Acc: 0.6887 | Val F1: 0.7836
Epoch  7/10 | Loss: 0.0095 | Val Acc: 0.6887 | Val F1: 0.7851
Epoch  8/10 | Loss: 0.0068 | Val Acc: 0.6887 | Val F1: 0.7836
Epoch  9/10 | Loss: 0.0053 | Val Acc: 0.6838 | Val F1: 0.7787
Epoch 10/10 | Loss: 0.0050 | Val Acc: 0.6814 | Val F1: 0.7833


In [20]:
seed = SEEDS[0]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model = FeedForwardNN(X_train.shape[1])

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [21]:
import copy

EPOCHS = 10

best_f1 = -1
best_epoch = -1
best_state = None

history = []

for epoch in range(EPOCHS):

    # -------- TRAIN --------
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    # -------- VALIDATION --------
    model.eval()

    with torch.no_grad():

        val_logits = model(X_val)

        val_probs = torch.sigmoid(val_logits).squeeze(1)

        val_preds = (
            val_probs >= THRESHOLD
        ).int().numpy()

    val_scores = evaluate(y_val, val_preds)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_accuracy": val_scores["accuracy"],
        "val_f1": val_scores["f1"],
    })

    # Save best model
    if val_scores["f1"] > best_f1:
        best_f1 = val_scores["f1"]
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:2d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_scores['accuracy']:.4f} | "
        f"Val F1: {val_scores['f1']:.4f}"
    )

print("\nBest epoch:", best_epoch)
print("Best validation F1:", round(best_f1, 4))

Epoch  1/10 | Loss: 0.6582 | Val Acc: 0.6912 | Val F1: 0.8158
Epoch  2/10 | Loss: 0.5305 | Val Acc: 0.7279 | Val F1: 0.8257
Epoch  3/10 | Loss: 0.2685 | Val Acc: 0.7255 | Val F1: 0.8164
Epoch  4/10 | Loss: 0.0900 | Val Acc: 0.7304 | Val F1: 0.8208
Epoch  5/10 | Loss: 0.0298 | Val Acc: 0.6985 | Val F1: 0.7868
Epoch  6/10 | Loss: 0.0153 | Val Acc: 0.6887 | Val F1: 0.7836
Epoch  7/10 | Loss: 0.0095 | Val Acc: 0.6887 | Val F1: 0.7851
Epoch  8/10 | Loss: 0.0068 | Val Acc: 0.6887 | Val F1: 0.7836
Epoch  9/10 | Loss: 0.0053 | Val Acc: 0.6838 | Val F1: 0.7787
Epoch 10/10 | Loss: 0.0050 | Val Acc: 0.6814 | Val F1: 0.7833

Best epoch: 2
Best validation F1: 0.8257


In [22]:
model.load_state_dict(best_state)

<All keys matched successfully>

In [23]:
architectures = [
    (256, 64),
    (128, 32),
    (64, 16),
]

In [24]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden2, 1)
        )

    def forward(self, x):
        return self.network(x)

In [25]:
import copy

results = []

for hidden1, hidden2 in architectures:

    seed = SEEDS[0]

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = FeedForwardNN(
        X_train.shape[1],
        hidden1,
        hidden2
    )

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    best_f1 = -1
    best_accuracy = -1
    best_epoch = -1
    best_state = None

    for epoch in range(10):

        # TRAIN
        model.train()

        for X_batch, y_batch in train_loader:

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = criterion(logits, y_batch)

            loss.backward()

            optimizer.step()

        # VALIDATE
        model.eval()

        with torch.no_grad():

            val_logits = model(X_val)

            val_probs = torch.sigmoid(
                val_logits
            ).squeeze(1)

            val_preds = (
                val_probs >= THRESHOLD
            ).int().numpy()

        scores = evaluate(y_val, val_preds)

        if scores["f1"] > best_f1:
            best_f1 = scores["f1"]
            best_accuracy = scores["accuracy"]
            best_epoch = epoch + 1
            best_state = copy.deepcopy(
                model.state_dict()
            )

    results.append({
        "Architecture": f"{hidden1}-{hidden2}",
        "Best Epoch": best_epoch,
        "Accuracy": best_accuracy,
        "F1": best_f1
    })

In [26]:
ffnn_results = pd.DataFrame(results)

ffnn_results

,Architecture,Best Epoch,Accuracy,F1
0,256-64,2,0.727941,0.825746
1,128-32,3,0.732843,0.825600
2,64-16,4,0.745098,0.831715


In [27]:
import copy

seed = SEEDS[0]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model = FeedForwardNN(
    X_train.shape[1],
    64,
    16
)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

best_f1 = -1
best_epoch = -1
best_state = None

for epoch in range(10):

    # TRAIN
    model.train()

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()

        optimizer.step()

    # VALIDATE
    model.eval()

    with torch.no_grad():

        val_logits = model(X_val)

        val_probs = torch.sigmoid(
            val_logits
        ).squeeze(1)

        val_preds = (
            val_probs >= THRESHOLD
        ).int().numpy()

    scores = evaluate(
        y_val,
        val_preds
    )

    if scores["f1"] > best_f1:
        best_f1 = scores["f1"]
        best_epoch = epoch + 1
        best_state = copy.deepcopy(
            model.state_dict()
        )

print("Best epoch:", best_epoch)
print("Best F1:", round(best_f1, 4))

Best epoch: 4
Best F1: 0.8317


In [29]:
model.load_state_dict(best_state)

<All keys matched successfully>

In [30]:
model.eval()

with torch.no_grad():

    val_logits = model(X_val)

    val_probs = torch.sigmoid(
        val_logits
    ).squeeze(1).numpy()

val_preds = (
    val_probs >= THRESHOLD
).astype(int)

In [31]:
ffnn_results = evaluate(
    y_val,
    val_preds
)

precision = precision_score(
    y_val,
    val_preds
)

recall = recall_score(
    y_val,
    val_preds
)

print(f"Accuracy:  {ffnn_results['accuracy']:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {ffnn_results['f1']:.4f}")

Accuracy:  0.7451
Precision: 0.7581
Recall:    0.9211
F1:        0.8317


In [32]:
cm = confusion_matrix(
    y_val,
    val_preds
)

print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print(f"\nTrue Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")

Confusion Matrix:
[[ 47  82]
 [ 22 257]]

True Negatives:  47
False Positives: 82
False Negatives: 22
True Positives:  257


In [34]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": 0.7402,
        "F1": 0.8374,
    },
    {
        "Model": "FFNN 64-16",
        "Accuracy": ffnn_results["accuracy"],
        "F1": ffnn_results["f1"],
    },
])

comparison

,Model,Accuracy,F1
0,Logistic Regression,0.740200,0.837400
1,FFNN 64-16,0.745098,0.831715


### Feed-Forward Neural Network Results

A feed-forward neural network was trained using the same TF-IDF sentence-pair
representation as the Logistic Regression model. Each sentence pair was
represented using the absolute difference and element-wise product of the two
TF-IDF vectors.

Three hidden-layer architectures were compared:

- 256-64
- 128-32
- 64-16

The larger networks overfit quickly, with training loss continuing to decrease
while validation performance deteriorated. The smaller 64-16 architecture
generalized best and achieved its best validation performance at epoch 4.

The final 64-16 FFNN achieved:

- Accuracy: 0.7451
- Precision: 0.7581
- Recall: 0.9211
- F1: 0.8317

Its confusion matrix contained 47 true negatives, 82 false positives,
22 false negatives, and 257 true positives.

Compared with Logistic Regression, the FFNN achieved slightly higher accuracy
and precision and produced fewer false positives. However, Logistic Regression
achieved higher recall and a slightly higher F1 score.

Overall, the smaller neural network generalized better than the larger
architectures, but Logistic Regression remained the strongest model by
validation F1.

In [36]:
import copy

seed_results = []
seed_states = {}

for seed in SEEDS:

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = FeedForwardNN(
        X_train.shape[1],
        64,
        16
    )

    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    best_f1 = -1
    best_epoch = -1
    best_accuracy = -1
    best_state = None

    for epoch in range(10):

        model.train()

        for X_batch, y_batch in train_loader:

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = criterion(logits, y_batch)

            loss.backward()

            optimizer.step()

        model.eval()

        with torch.no_grad():
            val_logits = model(X_val)
            val_probs = torch.sigmoid(val_logits).squeeze(1).numpy()

        val_preds = (
            val_probs >= THRESHOLD
        ).astype(int)

        scores = evaluate(
            y_val,
            val_preds
        )

        if scores["f1"] > best_f1:
            best_f1 = scores["f1"]
            best_accuracy = scores["accuracy"]
            best_epoch = epoch + 1
            best_state = copy.deepcopy(
                model.state_dict()
            )

    seed_states[seed] = best_state

    seed_results.append({
        "Seed": seed,
        "Best Epoch": best_epoch,
        "Accuracy": best_accuracy,
        "F1": best_f1
    })

In [37]:
seed_results_df = pd.DataFrame(seed_results)

seed_results_df

,Seed,Best Epoch,Accuracy,F1
0,42,4,0.745098,0.831715
1,43,5,0.742647,0.831461
2,44,3,0.727941,0.825197


In [38]:
print(
    f"Mean Accuracy: "
    f"{seed_results_df['Accuracy'].mean():.4f}"
)

print(
    f"Std Accuracy:  "
    f"{seed_results_df['Accuracy'].std():.4f}"
)

print(
    f"Mean F1:       "
    f"{seed_results_df['F1'].mean():.4f}"
)

print(
    f"Std F1:        "
    f"{seed_results_df['F1'].std():.4f}"
)

Mean Accuracy: 0.7386
Std Accuracy:  0.0093
Mean F1:       0.8295
Std F1:        0.0037


In [41]:
test_df = load_split("test")

y_test = test_df["label"].to_numpy()

print("Test:", test_df.shape)

Test: (1725, 4)


In [42]:
test_s1 = tfidf.transform(
    test_df["sentence1"]
)

test_s2 = tfidf.transform(
    test_df["sentence2"]
)

test_diff = abs(
    test_s1 - test_s2
)

test_product = test_s1.multiply(
    test_s2
)

X_test_sparse = hstack(
    [
        test_diff,
        test_product
    ]
).tocsr()

X_test = torch.tensor(
    X_test_sparse.toarray(),
    dtype=torch.float32
)

print("X_test:", X_test.shape)

X_test: torch.Size([1725, 19834])


In [44]:
test_s1 = tfidf.transform(test_df["sentence1"])
test_s2 = tfidf.transform(test_df["sentence2"])

test_diff = abs(test_s1 - test_s2)
test_product = test_s1.multiply(test_s2)

X_test_sparse = hstack(
    [test_diff, test_product]
).tocsr()

X_test = torch.tensor(
    X_test_sparse.toarray(),
    dtype=torch.float32
)

print("X_test:", X_test.shape)

X_test: torch.Size([1725, 19834])


In [48]:
test_results = []

for seed in SEEDS:

    model = FeedForwardNN(
        X_train.shape[1],
        64,
        16
    )

    model.load_state_dict(seed_states[seed])
    model.eval()

    with torch.no_grad():
        test_logits = model(X_test)

        test_probs = torch.sigmoid(
            test_logits
        ).squeeze(1).numpy()

    scores = save_predictions(
        test_df,
        test_probs,
        RESULTS_DIR
        / "group2_ffnn"
        / f"test_seed{seed}.csv"
    )

    test_results.append({
        "Seed": seed,
        "Accuracy": scores["accuracy"],
        "F1": scores["f1"],
    })

test_results_df = pd.DataFrame(test_results)
test_results_df

,Seed,Accuracy,F1
0,42,0.726377,0.814902
1,43,0.729275,0.816647
2,44,0.719420,0.814985


In [49]:
print(
    f"Mean Test Accuracy: "
    f"{test_results_df['Accuracy'].mean():.4f}"
)

print(
    f"Std Test Accuracy:  "
    f"{test_results_df['Accuracy'].std():.4f}"
)

print(
    f"Mean Test F1:       "
    f"{test_results_df['F1'].mean():.4f}"
)

print(
    f"Std Test F1:        "
    f"{test_results_df['F1'].std():.4f}"
)

Mean Test Accuracy: 0.7250
Std Test Accuracy:  0.0051
Mean Test F1:       0.8155
Std Test F1:        0.0010


In [ ]:
The final FFNN architecture used hidden layers of 64 and 16 units. The
architecture and training procedure were selected using only the validation
set.

The final model was evaluated using three random seeds: 42, 43, and 44.

Test results:

- Mean Accuracy: 0.7250
- Accuracy Std: 0.0051
- Mean F1: 0.8155
- F1 Std: 0.0010

The individual test F1 scores were very similar across the three seeds,
indicating stable performance.

Compared with Logistic Regression, the FFNN achieved slightly higher test
accuracy, while Logistic Regression achieved slightly higher test F1.

In [47]:
group2_summary = pd.DataFrame([
    {
        "model": "logistic_regression",
        "split": "test",
        "accuracy": 0.7154,
        "f1": 0.8164,
    },
    {
        "model": "ffnn_64_16_mean_3seeds",
        "split": "test",
        "accuracy": 0.7250,
        "f1": 0.8155,
    },
])

group2_summary.to_csv(
    "../results/group2_summary.csv",
    index=False
)

group2_summary

,model,split,accuracy,f1
0,logistic_regression,test,0.7154,0.8164
1,ffnn_64_16_mean_3seeds,test,0.7250,0.8155
